In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

PROJECT_ROOT = Path.cwd()
MOVIES_PATH = PROJECT_ROOT / "data" / "raw" / "ml-20m" / "movies.csv"
MODEL_DIR = PROJECT_ROOT / "models"

movies = pd.read_csv(MOVIES_PATH)
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)
movies["genres_clean"] = movies["genres_clean"].replace("(no genres listed)", "")

movies.head()

,movieId,title,genres,genres_clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


In [2]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies["genres_clean"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

# Map movieId to row index in movies DataFrame
content_movie_to_idx = pd.Series(movies.index, index=movies["movieId"]).to_dict()

TF-IDF matrix shape: (27278, 21)


In [3]:
def recommend_by_content(movie_id, n=10):
    if movie_id not in content_movie_to_idx:
        return "Movie not found."
    
    idx = content_movie_to_idx[movie_id]
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    # Exclude the movie itself
    sim_scores[idx] = -1
    
    top_indices = np.argpartition(sim_scores, -n)[-n:]
    top_indices = top_indices[np.argsort(sim_scores[top_indices])[::-1]]
    
    result = movies.iloc[top_indices][["movieId", "title", "genres"]].copy()
    result["content_similarity"] = np.round(sim_scores[top_indices], 4)
    return result

# Test with Toy Story (movieId = 1)
recommend_by_content(1, n=10)

,movieId,title,genres,content_similarity
24458,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,1.0
3922,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
21355,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0
3663,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
2209,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
10987,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
18274,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
3027,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
11871,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
24156,114552,"Boxtrolls, The (2014)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [4]:
def recommend_for_new_user(selected_genres, n=10):
    genre_query = " ".join(selected_genres)
    query_vec = tfidf.transform([genre_query])
    
    sim_scores = linear_kernel(query_vec, tfidf_matrix).flatten()
    
    top_indices = np.argpartition(sim_scores, -n)[-n:]
    top_indices = top_indices[np.argsort(sim_scores[top_indices])[::-1]]
    
    result = movies.iloc[top_indices][["movieId", "title", "genres"]].copy()
    result["match_score"] = np.round(sim_scores[top_indices], 4)
    return result

# Save TF-IDF vectorizer for deployment
joblib.dump(tfidf, MODEL_DIR / "tfidf_vectorizer.joblib")

# Test for a new user who likes Action and Sci-Fi
recommend_for_new_user(["Action", "Sci-Fi"], n=10)

,movieId,title,genres,match_score
23318,110984,Age of Dinosaurs (2013),Action|Sci-Fi,1.0
12544,58722,Terror of Mechagodzilla (Mekagojira no gyakush...,Action|Sci-Fi,1.0
15872,80505,2012: Supernova (2009),Action|Sci-Fi,1.0
4998,5094,Rollerball (2002),Action|Sci-Fi,1.0
4485,4580,Cyborg (1989),Action|Sci-Fi,1.0
20363,100010,Battle of Los Angeles (2011),Action|Sci-Fi,1.0
6473,6583,"Blood of Heroes, The (Salute of the Jugger, Th...",Action|Sci-Fi,1.0
4596,4691,Def-Con 4 (1985),Action|Sci-Fi,1.0
11898,53370,Godzilla and Mothra: The Battle for Earth (Goj...,Action|Sci-Fi,1.0
2448,2533,Escape from the Planet of the Apes (1971),Action|Sci-Fi,1.0
